# 03 · GSE65391 · RNA_array · batch check

Reads `genes.rds`, `metadata.rds`. Writes `data/run_artifacts/GSE65391/expression.rds`.

The arrays were run in two batches. The same healthy blood samples were run in both
(`set = Technical_Replicate`). A pair of arrays from one blood sample differs only by batch and noise.

**Test.** For each expressed gene: batch 2 minus batch 1 across the replicate pairs, and a paired
t statistic. **A residual batch effect** would show as many genes with |t| > 5.

In [1]:
source("../src/paths.R")
g    <- readRDS(art("GSE65391", "genes.rds"))
meta <- readRDS(art("GSE65391", "metadata.rds"))
E    <- g$E[g$expressed, rownames(meta)]
h <- meta[meta$disease == "Healthy", ]
pairs <- split(rownames(h), paste(h$subject, h$visit))
pairs <- pairs[vapply(pairs, function(p) setequal(h[p, "batch"], c("1", "2")), logical(1))]
b1 <- vapply(pairs, function(p) p[h[p, "batch"] == "1"][1], "")
b2 <- vapply(pairs, function(p) p[h[p, "batch"] == "2"][1], "")
d  <- E[, b2] - E[, b1]
t_paired <- rowMeans(d) / (apply(d, 1, sd) / sqrt(ncol(d)))
c(replicate_pairs = length(pairs), genes = nrow(d),
  median_abs_shift = round(median(abs(rowMeans(d))), 3),
  genes_abs_t_above_5 = sum(abs(t_paired) > 5))

replicate_pairs               genes    median_abs_shift genes_abs_t_above_5 
             23.000            8825.000               0.048               0.000

**Result.** 23 replicate pairs. Median absolute shift 0.048 log2 units; 0 of 8,825 genes with
|t| > 5. No residual batch effect.

**Test.** Mean correlation of the two arrays of a replicate pair, against the mean correlation of
arrays from different children in the same batch. **A residual batch effect** would make these similar.

In [2]:
cc <- cor(E[, b1])
c(replicate_pairs = round(mean(diag(cor(E[, b1], E[, b2]))), 3),
  different_children = round(mean(cc[upper.tri(cc)]), 3))

replicate_pairs different_children 
             0.969              0.928

**Result.** Replicate pairs correlate at 0.969, arrays of different children at 0.928.

The technical replicates are second copies of samples already in the table; they are dropped.

In [3]:
keep <- meta$set != "Technical_Replicate"
saveRDS(list(E = g$E[, keep], expressed = g$expressed, meta = meta[keep, ]), art("GSE65391", "expression.rds"))
table(disease = meta$disease[keep])

disease
Healthy     SLE 
     48     924 

**Result.** 972 samples remain: 48 healthy, 924 SLE.